In [2]:
import duckdb
import pandas as pd
from pathlib import Path
from datetime import datetime

# Paths — everything relative to the notebook location
PROJECT_ROOT = Path("..").resolve()
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
DB_PATH      = PROJECT_ROOT / "nhs_ae.duckdb"

# Confirm the raw files are visible
csv_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  {f.name}")

Found 5 CSV files:
  February-2026-CSV-Dl8t54.csv
  Monthly-AE-December-2025-revised-g9Hy6.csv
  Monthly-AE-January-2026-revised-3dker.csv
  Monthly-AE-March-2026-revised-flkg42.csv
  Monthly-AE-Nov-25-CSV-revised.csv


In [3]:
# Cell 2 — Connect to DuckDB (creates the file if it doesn't exist)
con = duckdb.connect(str(DB_PATH))
print(f"Connected to: {DB_PATH}")
print(f"DuckDB version: {duckdb.__version__}")

Connected to: C:\Users\q8x7u\Documents\cv\CV\NHS project\nhs-ae-pipeline\nhs_ae.duckdb
DuckDB version: 1.5.5


In [4]:
# Cell 3 — Create schemas and tables
schema_statements = [
    "CREATE SCHEMA IF NOT EXISTS raw",
    "CREATE SCHEMA IF NOT EXISTS reporting",
    "CREATE SCHEMA IF NOT EXISTS audit",

    """CREATE TABLE IF NOT EXISTS raw.ae_monthly (
        load_id       INTEGER,
        source_file   VARCHAR,
        loaded_at     TIMESTAMP,
        row_number    INTEGER,
        period_raw    VARCHAR,
        org_code      VARCHAR,
        parent_org    VARCHAR,
        org_name      VARCHAR,
        type1_attendances       VARCHAR,
        type2_attendances       VARCHAR,
        other_attendances       VARCHAR,
        booked_appts_type1      VARCHAR,
        booked_appts_type2      VARCHAR,
        booked_appts_other      VARCHAR,
        over_4h_type1           VARCHAR,
        over_4h_type2           VARCHAR,
        over_4h_other           VARCHAR,
        over_4h_booked_type1    VARCHAR,
        over_4h_booked_type2    VARCHAR,
        over_4h_booked_other    VARCHAR,
        wait_4_12h_dta          VARCHAR,
        wait_12plus_dta         VARCHAR,
        emergency_admissions_type1  VARCHAR,
        emergency_admissions_type2  VARCHAR,
        emergency_admissions_other  VARCHAR,
        other_emergency_admissions  VARCHAR
    )""",

    """CREATE TABLE IF NOT EXISTS reporting.ae_monthly (
        period                      DATE,
        org_code                    VARCHAR,
        parent_org                  VARCHAR,
        org_name                    VARCHAR,
        type1_attendances           INTEGER,
        type2_attendances           INTEGER,
        other_attendances           INTEGER,
        total_attendances           INTEGER,
        over_4h_type1               INTEGER,
        over_4h_type2               INTEGER,
        over_4h_other               INTEGER,
        total_over_4h               INTEGER,
        within_4h_type1             INTEGER,
        four_hour_pct_type1         DOUBLE,
        wait_4_12h_dta              INTEGER,
        wait_12plus_dta             INTEGER,
        emergency_admissions_type1  INTEGER,
        emergency_admissions_type2  INTEGER,
        emergency_admissions_other  INTEGER,
        other_emergency_admissions  INTEGER,
        source_file                 VARCHAR,
        loaded_at                   TIMESTAMP
    )""",

    """CREATE TABLE IF NOT EXISTS audit.refresh_log (
        load_id             INTEGER,
        source_file         VARCHAR,
        publication_month   VARCHAR,
        started_at          TIMESTAMP,
        completed_at        TIMESTAMP,
        rows_loaded         INTEGER,
        rows_rejected       INTEGER,
        status              VARCHAR,
        notes               TEXT
    )""",

    """CREATE TABLE IF NOT EXISTS audit.quality_issues (
        issue_id        INTEGER,
        load_id         INTEGER,
        check_name      VARCHAR,
        severity        VARCHAR,
        description     TEXT,
        affected_rows   INTEGER,
        detected_at     TIMESTAMP
    )"""
]

for stmt in schema_statements:
    con.execute(stmt)

# Verify tables created
tables = con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema IN ('raw', 'reporting', 'audit')
    ORDER BY table_schema, table_name
""").df()

print("Tables created:")
print(tables.to_string(index=False))

Tables created:
table_schema     table_name
       audit quality_issues
       audit    refresh_log
         raw     ae_monthly
   reporting     ae_monthly


In [6]:
# Cell 4 — Load raw data (corrected)

con.execute("DELETE FROM raw.ae_monthly")
con.execute("DELETE FROM audit.refresh_log")

COLUMN_MAP = {
    "Period"                                                    : "period_raw",
    "Org Code"                                                  : "org_code",
    "Parent Org"                                                : "parent_org",
    "Org name"                                                  : "org_name",
    "A&E attendances Type 1"                                    : "type1_attendances",
    "A&E attendances Type 2"                                    : "type2_attendances",
    "A&E attendances Other A&E Department"                      : "other_attendances",
    "A&E attendances Booked Appointments Type 1"                : "booked_appts_type1",
    "A&E attendances Booked Appointments Type 2"                : "booked_appts_type2",
    "A&E attendances Booked Appointments Other Department"      : "booked_appts_other",
    "Attendances over 4hrs Type 1"                             : "over_4h_type1",
    "Attendances over 4hrs Type 2"                             : "over_4h_type2",
    "Attendances over 4hrs Other Department"                   : "over_4h_other",
    "Attendances over 4hrs Booked Appointments Type 1"         : "over_4h_booked_type1",
    "Attendances over 4hrs Booked Appointments Type 2"         : "over_4h_booked_type2",
    "Attendances over 4hrs Booked Appointments Other Department": "over_4h_booked_other",
    "Patients who have waited 4-12 hs from DTA to admission"   : "wait_4_12h_dta",
    "Patients who have waited 12+ hrs from DTA to admission"   : "wait_12plus_dta",
    "Emergency admissions via A&E - Type 1"                    : "emergency_admissions_type1",
    "Emergency admissions via A&E - Type 2"                    : "emergency_admissions_type2",
    "Emergency admissions via A&E - Other A&E department"      : "emergency_admissions_other",
    "Other emergency admissions"                               : "other_emergency_admissions",
}

COL_ORDER = [
    "load_id", "source_file", "loaded_at", "row_number",
    "period_raw", "org_code", "parent_org", "org_name",
    "type1_attendances", "type2_attendances", "other_attendances",
    "booked_appts_type1", "booked_appts_type2", "booked_appts_other",
    "over_4h_type1", "over_4h_type2", "over_4h_other",
    "over_4h_booked_type1", "over_4h_booked_type2", "over_4h_booked_other",
    "wait_4_12h_dta", "wait_12plus_dta",
    "emergency_admissions_type1", "emergency_admissions_type2",
    "emergency_admissions_other", "other_emergency_admissions"
]

def get_next_load_id(con):
    return con.execute(
        "SELECT COALESCE(MAX(load_id), 0) + 1 FROM audit.refresh_log"
    ).fetchone()[0]

def load_file(con, filepath):
    source_file = filepath.name
    load_id     = get_next_load_id(con)
    started_at  = datetime.now()

    con.execute("""
        INSERT INTO audit.refresh_log (load_id, source_file, started_at, status)
        VALUES (?, ?, ?, 'running')
    """, [load_id, source_file, started_at])

    try:
        df = pd.read_csv(filepath, dtype=str)
        df.columns = df.columns.str.strip()

        missing = [c for c in COLUMN_MAP if c not in df.columns]
        if missing:
            print(f"  ⚠️  Missing columns in {source_file}: {missing}")

        df = df.rename(columns=COLUMN_MAP)
        df = df.dropna(how="all")

        df["load_id"]     = load_id
        df["source_file"] = source_file
        df["loaded_at"]   = datetime.now()
        df["row_number"]  = range(1, len(df) + 1)

        # Extract publication month from period_raw
        sample_period = df["period_raw"].dropna().iloc[0] if "period_raw" in df.columns else ""
        parts = str(sample_period).split("-")
        pub_month = f"{parts[1]}-{parts[2]}" if len(parts) >= 3 else "UNKNOWN"

        # Reorder BEFORE inserting
        df = df[COL_ORDER]

        # Idempotent delete then insert
        con.execute("DELETE FROM raw.ae_monthly WHERE source_file = ?", [source_file])
        con.execute("INSERT INTO raw.ae_monthly SELECT * FROM df")
        rows_loaded = len(df)

        con.execute("""
            UPDATE audit.refresh_log
            SET completed_at = ?, rows_loaded = ?, publication_month = ?, status = 'success'
            WHERE load_id = ?
        """, [datetime.now(), rows_loaded, pub_month, load_id])

        print(f"  ✅ {source_file}: {rows_loaded} rows (load_id={load_id}, month={pub_month})")
        return load_id

    except Exception as e:
        con.execute("""
            UPDATE audit.refresh_log SET status = 'failed', notes = ? WHERE load_id = ?
        """, [str(e), load_id])
        print(f"  ❌ {source_file}: FAILED — {e}")
        raise

print("Loading files into raw.ae_monthly...\n")
for filepath in csv_files:
    load_file(con, filepath)

print("\nRow counts by source file:")
print(con.execute("""
    SELECT source_file, COUNT(*) AS rows
    FROM raw.ae_monthly
    GROUP BY source_file
    ORDER BY source_file
""").df().to_string(index=False))

Loading files into raw.ae_monthly...

  ✅ February-2026-CSV-Dl8t54.csv: 198 rows (load_id=1, month=FEBRUARY-2026)
  ✅ Monthly-AE-December-2025-revised-g9Hy6.csv: 198 rows (load_id=2, month=DECEMBER-2025)
  ✅ Monthly-AE-January-2026-revised-3dker.csv: 198 rows (load_id=3, month=JANUARY-2026)
  ✅ Monthly-AE-March-2026-revised-flkg42.csv: 200 rows (load_id=4, month=MARCH-2026)
  ✅ Monthly-AE-Nov-25-CSV-revised.csv: 198 rows (load_id=5, month=NOVEMBER-2025)

Row counts by source file:
                               source_file  rows
              February-2026-CSV-Dl8t54.csv   198
Monthly-AE-December-2025-revised-g9Hy6.csv   198
 Monthly-AE-January-2026-revised-3dker.csv   198
  Monthly-AE-March-2026-revised-flkg42.csv   200
         Monthly-AE-Nov-25-CSV-revised.csv   198


In [7]:
# Cell 5 — Transform raw data into reporting.ae_monthly

import re

def parse_period(period_raw):
    """Convert 'MSitAE-MARCH-2026' to a proper date (first of the month)."""
    try:
        # Extract month and year from the string
        parts = str(period_raw).split("-")
        month_str = parts[1]   # e.g. MARCH
        year_str  = parts[2]   # e.g. 2026
        return pd.to_datetime(f"1 {month_str} {year_str}", format="%d %B %Y").date()
    except Exception:
        return None

def safe_int(value):
    """Convert to integer, returning None if not possible."""
    try:
        return int(str(value).replace(",", "").strip())
    except (ValueError, TypeError):
        return None

# Read raw data
raw_df = con.execute("SELECT * FROM raw.ae_monthly").df()
print(f"Raw rows to transform: {len(raw_df)}")

# Parse period
raw_df["period"] = raw_df["period_raw"].apply(parse_period)
unparsed = raw_df["period"].isna().sum()
if unparsed > 0:
    print(f"  ⚠️  {unparsed} rows could not have period parsed")

# Cast attendance columns to integer
int_cols = [
    "type1_attendances", "type2_attendances", "other_attendances",
    "booked_appts_type1", "booked_appts_type2", "booked_appts_other",
    "over_4h_type1", "over_4h_type2", "over_4h_other",
    "over_4h_booked_type1", "over_4h_booked_type2", "over_4h_booked_other",
    "wait_4_12h_dta", "wait_12plus_dta",
    "emergency_admissions_type1", "emergency_admissions_type2",
    "emergency_admissions_other", "other_emergency_admissions"
]
for col in int_cols:
    raw_df[col] = raw_df[col].apply(safe_int)

# Derive totals — do not trust any pre-calculated figures from the source
raw_df["total_attendances"] = (
    raw_df["type1_attendances"].fillna(0) +
    raw_df["type2_attendances"].fillna(0) +
    raw_df["other_attendances"].fillna(0)
)

# Within 4 hours Type 1 = Type 1 attendances minus those over 4 hours
raw_df["within_4h_type1"] = (
    raw_df["type1_attendances"].fillna(0) -
    raw_df["over_4h_type1"].fillna(0)
)

# Total over 4 hours across all types
raw_df["total_over_4h"] = (
    raw_df["over_4h_type1"].fillna(0) +
    raw_df["over_4h_type2"].fillna(0) +
    raw_df["over_4h_other"].fillna(0)
)

# 4-hour performance % for Type 1
# Avoid division by zero with where()
raw_df["four_hour_pct_type1"] = (
    (raw_df["within_4h_type1"] / raw_df["type1_attendances"].replace(0, None)) * 100
).round(1)

# Select and order reporting columns
reporting_df = raw_df[[
    "period", "org_code", "parent_org", "org_name",
    "type1_attendances", "type2_attendances", "other_attendances",
    "total_attendances",
    "over_4h_type1", "over_4h_type2", "over_4h_other",
    "total_over_4h", "within_4h_type1", "four_hour_pct_type1",
    "wait_4_12h_dta", "wait_12plus_dta",
    "emergency_admissions_type1", "emergency_admissions_type2",
    "emergency_admissions_other", "other_emergency_admissions",
    "source_file", "loaded_at"
]].copy()

# Load into reporting table (clear first for clean reload)
con.execute("DELETE FROM reporting.ae_monthly")
con.execute("INSERT INTO reporting.ae_monthly SELECT * FROM reporting_df")

# Verify
result = con.execute("""
    SELECT
        period,
        COUNT(*)            AS providers,
        SUM(type1_attendances)  AS total_type1,
        ROUND(AVG(four_hour_pct_type1), 1) AS avg_4h_pct
    FROM reporting.ae_monthly
    GROUP BY period
    ORDER BY period
""").df()

print("\nReporting table summary:")
print(result.to_string(index=False))

Raw rows to transform: 992
  ⚠️  4 rows could not have period parsed

Reporting table summary:
    period  providers  total_type1  avg_4h_pct
2025-11-01        197    1420033.0        59.9
2025-12-01        197    1407641.0        59.1
2026-01-01        197    1405035.0        56.5
2026-02-01        198    2543660.0        58.8
2026-03-01        199    1451010.0        63.6
       NaT          4    5683719.0        60.1


In [8]:
# Cell 6 — Investigate anomalies

# --- Issue 1: What are the 4 unparsed period rows? ---
print("=== Unparsed period rows ===")
unparsed_df = raw_df[raw_df["period"].isna()][
    ["period_raw", "org_code", "org_name", "source_file"]
]
print(unparsed_df.to_string(index=False))

# --- Issue 2: February outliers ---
print("\n=== February 2026: top 10 providers by Type 1 attendances ===")
feb_df = raw_df[raw_df["period"] == pd.Timestamp("2026-02-01").date()].copy()
feb_top = (feb_df[["org_code", "org_name", "type1_attendances"]]
           .sort_values("type1_attendances", ascending=False)
           .head(10))
print(feb_top.to_string(index=False))

# Compare the same providers across all months
print("\n=== Same providers — Type 1 attendances across all months ===")
top_orgs = feb_top["org_code"].tolist()
comparison = raw_df[raw_df["org_code"].isin(top_orgs)][
    ["org_code", "org_name", "period", "type1_attendances"]
].sort_values(["org_code", "period"])
print(comparison.to_string(index=False))

=== Unparsed period rows ===
period_raw org_code org_name                                source_file
     TOTAL    TOTAL    TOTAL Monthly-AE-December-2025-revised-g9Hy6.csv
     TOTAL    TOTAL    TOTAL  Monthly-AE-January-2026-revised-3dker.csv
     TOTAL    TOTAL    TOTAL   Monthly-AE-March-2026-revised-flkg42.csv
     TOTAL    TOTAL    TOTAL          Monthly-AE-Nov-25-CSV-revised.csv

=== February 2026: top 10 providers by Type 1 attendances ===
org_code                                             org_name  type1_attendances
   TOTAL                                                TOTAL            1271830
     RRK UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION TRUST              30640
     RAJ             MID AND SOUTH ESSEX NHS FOUNDATION TRUST              24311
     RAL               ROYAL FREE LONDON NHS FOUNDATION TRUST              24307
     R0A           MANCHESTER UNIVERSITY NHS FOUNDATION TRUST              24273
     RYR     UNIVERSITY HOSPITALS SUSSEX NHS FOUNDATION TRUST 

In [9]:
# Cell 7 — Rebuild reporting table excluding TOTAL rows

# Check how many TOTAL rows exist across all files
total_rows = raw_df[raw_df["org_code"] == "TOTAL"]
print(f"TOTAL rows found: {len(total_rows)} (one per file — will be excluded)")

# Rebuild reporting_df excluding TOTAL rows
reporting_df_clean = reporting_df[
    reporting_df["org_code"] != "TOTAL"
].copy()

# Also drop any rows where period is NaT (just in case)
reporting_df_clean = reporting_df_clean[
    reporting_df_clean["period"].notna()
].copy()

print(f"Rows before exclusion: {len(reporting_df)}")
print(f"Rows after exclusion:  {len(reporting_df_clean)}")

# Reload reporting table
con.execute("DELETE FROM reporting.ae_monthly")
con.execute("INSERT INTO reporting.ae_monthly SELECT * FROM reporting_df_clean")

# Verify — February should now match other months
result = con.execute("""
    SELECT
        period,
        COUNT(*)                            AS providers,
        SUM(type1_attendances)              AS total_type1,
        ROUND(AVG(four_hour_pct_type1), 1)  AS avg_4h_pct
    FROM reporting.ae_monthly
    GROUP BY period
    ORDER BY period
""").df()

print("\nReporting table summary (TOTAL rows excluded):")
print(result.to_string(index=False))

TOTAL rows found: 5 (one per file — will be excluded)
Rows before exclusion: 992
Rows after exclusion:  987

Reporting table summary (TOTAL rows excluded):
    period  providers  total_type1  avg_4h_pct
2025-11-01        197    1420033.0        59.9
2025-12-01        197    1407641.0        59.1
2026-01-01        197    1405035.0        56.5
2026-02-01        197    1271830.0        58.8
2026-03-01        199    1451010.0        63.6


In [10]:
# Cell 8 — Data quality checks

from datetime import datetime

issue_id_counter = [1]  # mutable counter

def log_issue(con, load_id, check_name, severity, description, affected_rows):
    con.execute("""
        INSERT INTO audit.quality_issues
            (issue_id, load_id, check_name, severity, description, affected_rows, detected_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, [issue_id_counter[0], load_id, check_name, severity, description,
          affected_rows, datetime.now()])
    issue_id_counter[0] += 1

def run_checks(con, load_id=None):
    """
    Run all quality checks against reporting.ae_monthly.
    load_id=None checks all data; pass a specific load_id to check one file.
    """
    where = f"WHERE load_id = {load_id}" if load_id else ""
    issues = []

    # ── Check 1: Missing org codes ────────────────────────────────────────
    n = con.execute(f"""
        SELECT COUNT(*) FROM reporting.ae_monthly {where}
        AND (org_code IS NULL OR TRIM(org_code) = '')
    """ if where else """
        SELECT COUNT(*) FROM reporting.ae_monthly
        WHERE org_code IS NULL OR TRIM(org_code) = ''
    """).fetchone()[0]

    if n > 0:
        desc = f"{n} rows have a missing or blank org_code"
        log_issue(con, load_id or 0, "MISSING_ORG_CODE", "ERROR", desc, n)
        issues.append(("ERROR", "MISSING_ORG_CODE", desc))
    else:
        issues.append(("PASS", "MISSING_ORG_CODE", "All rows have an org_code"))

    # ── Check 2: Duplicate (period, org_code) combinations ───────────────
    n = con.execute("""
        SELECT COUNT(*) FROM (
            SELECT period, org_code, COUNT(*) AS n
            FROM reporting.ae_monthly
            GROUP BY period, org_code
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    if n > 0:
        desc = f"{n} (period, org_code) combinations appear more than once"
        log_issue(con, load_id or 0, "DUPLICATE_ORG_MONTH", "ERROR", desc, n)
        issues.append(("ERROR", "DUPLICATE_ORG_MONTH", desc))
    else:
        issues.append(("PASS", "DUPLICATE_ORG_MONTH", "No duplicate (period, org_code) pairs"))

    # ── Check 3: Negative attendance counts ──────────────────────────────
    n = con.execute("""
        SELECT COUNT(*) FROM reporting.ae_monthly
        WHERE type1_attendances < 0
           OR type2_attendances < 0
           OR other_attendances < 0
           OR over_4h_type1     < 0
    """).fetchone()[0]

    if n > 0:
        desc = f"{n} rows contain negative attendance counts"
        log_issue(con, load_id or 0, "NEGATIVE_COUNTS", "ERROR", desc, n)
        issues.append(("ERROR", "NEGATIVE_COUNTS", desc))
    else:
        issues.append(("PASS", "NEGATIVE_COUNTS", "No negative attendance counts"))

    # ── Check 4: 4-hour performance outside 0–100% ───────────────────────
    n = con.execute("""
        SELECT COUNT(*) FROM reporting.ae_monthly
        WHERE four_hour_pct_type1 IS NOT NULL
          AND (four_hour_pct_type1 < 0 OR four_hour_pct_type1 > 100)
    """).fetchone()[0]

    if n > 0:
        desc = f"{n} rows have four_hour_pct_type1 outside the range 0–100"
        log_issue(con, load_id or 0, "PERFORMANCE_OUT_OF_RANGE", "WARNING", desc, n)
        issues.append(("WARNING", "PERFORMANCE_OUT_OF_RANGE", desc))
    else:
        issues.append(("PASS", "PERFORMANCE_OUT_OF_RANGE", "All performance values within 0–100"))

    # ── Check 5: Within-4h cannot exceed total attendances ───────────────
    n = con.execute("""
        SELECT COUNT(*) FROM reporting.ae_monthly
        WHERE within_4h_type1 > type1_attendances
    """).fetchone()[0]

    if n > 0:
        desc = f"{n} rows where within_4h_type1 exceeds type1_attendances"
        log_issue(con, load_id or 0, "WITHIN_4H_EXCEEDS_TOTAL", "ERROR", desc, n)
        issues.append(("ERROR", "WITHIN_4H_EXCEEDS_TOTAL", desc))
    else:
        issues.append(("PASS", "WITHIN_4H_EXCEEDS_TOTAL", "within_4h_type1 is within bounds"))

    # ── Print summary ─────────────────────────────────────────────────────
    print(f"\nQuality check results ({datetime.now().strftime('%Y-%m-%d %H:%M')})")
    print("-" * 65)
    for status, check, desc in issues:
        icon = "✅" if status == "PASS" else ("❌" if status == "ERROR" else "⚠️")
        print(f"{icon}  {check:<35} {desc}")

    errors   = sum(1 for s, _, _ in issues if s == "ERROR")
    warnings = sum(1 for s, _, _ in issues if s == "WARNING")
    print("-" * 65)
    print(f"   {errors} error(s), {warnings} warning(s), "
          f"{len(issues)-errors-warnings} passed")
    return issues

# Run checks on the full reporting table
all_issues = run_checks(con)


Quality check results (2026-09-20 14:03)
-----------------------------------------------------------------
✅  MISSING_ORG_CODE                    All rows have an org_code
✅  DUPLICATE_ORG_MONTH                 No duplicate (period, org_code) pairs
✅  NEGATIVE_COUNTS                     No negative attendance counts
✅  PERFORMANCE_OUT_OF_RANGE            All performance values within 0–100
✅  WITHIN_4H_EXCEEDS_TOTAL             within_4h_type1 is within bounds
-----------------------------------------------------------------
   0 error(s), 0 warning(s), 5 passed


In [11]:
# Cell 9 — Prove checks catch bad data

print("Inserting a deliberately bad row...")
con.execute("""
    INSERT INTO reporting.ae_monthly
        (period, org_code, org_name, type1_attendances, over_4h_type1,
         within_4h_type1, four_hour_pct_type1, source_file, loaded_at)
    VALUES
        ('2026-03-01', NULL, 'Test Trust', -999, 0, 0, 150.0, 'TEST', NOW())
""")

print("Running checks against data including bad row:\n")
run_checks(con)

# Clean up the bad row
con.execute("DELETE FROM reporting.ae_monthly WHERE source_file = 'TEST'")
con.execute("DELETE FROM audit.quality_issues WHERE load_id = 0")
print("\n✅ Bad row removed — table restored to clean state")

Inserting a deliberately bad row...
Running checks against data including bad row:


Quality check results (2026-09-20 14:04)
-----------------------------------------------------------------
❌  MISSING_ORG_CODE                    1 rows have a missing or blank org_code
✅  DUPLICATE_ORG_MONTH                 No duplicate (period, org_code) pairs
❌  NEGATIVE_COUNTS                     1 rows contain negative attendance counts
⚠️  PERFORMANCE_OUT_OF_RANGE            1 rows have four_hour_pct_type1 outside the range 0–100
❌  WITHIN_4H_EXCEEDS_TOTAL             1 rows where within_4h_type1 exceeds type1_attendances
-----------------------------------------------------------------
   3 error(s), 1 warning(s), 1 passed

✅ Bad row removed — table restored to clean state


In [12]:
# Cell 10 — Write the data dictionary and view the refresh log

# ── Data dictionary as a markdown file ───────────────────────────────────
data_dict = """# NHS A&E Pipeline — Data Dictionary

**Source:** NHS England A&E Attendances and Emergency Admissions
**URL:** https://www.england.nhs.uk/statistics/statistical-work-areas/ae-waiting-times-and-activity/
**Update frequency:** Monthly (published on the 2nd Thursday of each month, covering the previous month)
**Downloaded by:** Manual download from NHS England website

---

## Table: reporting.ae_monthly

| Field | Type | Description | Derived from | Notes |
|---|---|---|---|---|
| period | DATE | First day of the reporting month e.g. 2026-03-01 | period_raw column in source | Source format is MSitAE-MARCH-2026; parsed in pipeline |
| org_code | VARCHAR | NHS ODS provider code e.g. RRK | Org Code | Used to identify providers consistently across months |
| parent_org | VARCHAR | NHS England regional team | Parent Org | May change if provider moves region |
| org_name | VARCHAR | Provider name as published | Org name | May change following mergers or reconfigurations |
| type1_attendances | INTEGER | Attendances at Type 1 (major) A&E departments | A&E attendances Type 1 | Excludes UTCs and walk-in centres |
| type2_attendances | INTEGER | Attendances at Type 2 (single specialty) A&E departments | A&E attendances Type 2 | |
| other_attendances | INTEGER | Attendances at other departments including UTCs | A&E attendances Other A&E Department | Definition expanded over time — see NHS guidance |
| total_attendances | INTEGER | Sum of Type 1, Type 2 and other attendances | Derived | Calculated in pipeline; not taken from source TOTAL row |
| over_4h_type1 | INTEGER | Type 1 attendances not discharged, admitted or transferred within 4 hours | Attendances over 4hrs Type 1 | |
| over_4h_type2 | INTEGER | Type 2 attendances over 4 hours | Attendances over 4hrs Type 2 | |
| over_4h_other | INTEGER | Other department attendances over 4 hours | Attendances over 4hrs Other Department | |
| total_over_4h | INTEGER | Sum of all attendances over 4 hours | Derived | Calculated in pipeline |
| within_4h_type1 | INTEGER | Type 1 attendances seen within 4 hours | Derived | type1_attendances minus over_4h_type1 |
| four_hour_pct_type1 | DOUBLE | Percentage of Type 1 attendances within 4 hours | Derived | within_4h_type1 / type1_attendances * 100; national target is 95% |
| wait_4_12h_dta | INTEGER | Patients waiting 4-12 hours from decision to admit to admission | Patients who have waited 4-12 hs from DTA to admission | DTA = Decision To Admit |
| wait_12plus_dta | INTEGER | Patients waiting 12 or more hours from decision to admit | Patients who have waited 12+ hrs from DTA to admission | Key measure of hospital flow pressure |
| emergency_admissions_type1 | INTEGER | Emergency admissions arriving via Type 1 A&E | Emergency admissions via A&E - Type 1 | |
| emergency_admissions_type2 | INTEGER | Emergency admissions arriving via Type 2 A&E | Emergency admissions via A&E - Type 2 | |
| emergency_admissions_other | INTEGER | Emergency admissions arriving via other departments | Emergency admissions via A&E - Other A&E department | |
| other_emergency_admissions | INTEGER | Emergency admissions not via A&E | Other emergency admissions | |
| source_file | VARCHAR | Filename this row was loaded from | Pipeline metadata | Use to trace back to original downloaded file |
| loaded_at | TIMESTAMP | When this row was inserted into the pipeline | Pipeline metadata | |

---

## Table: audit.refresh_log

Records every file load attempt.

| Field | Description |
|---|---|
| load_id | Unique identifier for each load run |
| source_file | Filename loaded |
| publication_month | Month covered by the file e.g. MARCH-2026 |
| started_at | When the load began |
| completed_at | When the load finished |
| rows_loaded | Rows inserted into raw.ae_monthly |
| status | success, failed or running |
| notes | Error message if status is failed |

---

## Table: audit.quality_issues

Records every quality problem detected.

| Field | Description |
|---|---|
| issue_id | Unique identifier |
| load_id | Links to refresh_log |
| check_name | Name of the check that fired |
| severity | ERROR or WARNING |
| description | Human-readable description of the problem |
| affected_rows | Number of rows affected |
| detected_at | When the check ran |

---

## Reporting considerations

NHS England notes several factors that affect interpretation of this data:

- **Provider reconfigurations:** Mergers, splits and reclassifications mean that
  changes between months do not always reflect changes in patient activity.
- **Late submissions:** Some providers submit revised figures after initial publication.
  Files marked revised in the filename contain corrected data.
- **February:** Shorter month means lower raw attendance counts are expected
  and should not be interpreted as a drop in demand.
- **Type 3 / Other departments:** The definition of this category has changed
  over time. Long-run comparisons should account for this.
- **TOTAL row:** Each source file contains a pre-calculated TOTAL summary row
  with org_code = TOTAL. This row is excluded from the reporting table.
"""

dict_path = PROJECT_ROOT / "docs" / "data_dictionary.md"
with open(dict_path, "w", encoding="utf-8") as f:
    f.write(data_dict)
print(f"Data dictionary written to: {dict_path}")

# ── View the refresh log ──────────────────────────────────────────────────
print("\nRefresh log:")
refresh_log = con.execute("""
    SELECT
        load_id,
        source_file,
        publication_month,
        started_at,
        rows_loaded,
        status
    FROM audit.refresh_log
    ORDER BY load_id
""").df()
print(refresh_log.to_string(index=False))

Data dictionary written to: C:\Users\q8x7u\Documents\cv\CV\NHS project\nhs-ae-pipeline\docs\data_dictionary.md

Refresh log:
 load_id                                source_file publication_month                 started_at  rows_loaded  status
       1               February-2026-CSV-Dl8t54.csv     FEBRUARY-2026 2026-09-20 13:58:26.430151          198 success
       2 Monthly-AE-December-2025-revised-g9Hy6.csv     DECEMBER-2025 2026-09-20 13:58:26.458256          198 success
       3  Monthly-AE-January-2026-revised-3dker.csv      JANUARY-2026 2026-09-20 13:58:26.489229          198 success
       4   Monthly-AE-March-2026-revised-flkg42.csv        MARCH-2026 2026-09-20 13:58:26.510483          200 success
       5          Monthly-AE-Nov-25-CSV-revised.csv     NOVEMBER-2025 2026-09-20 13:58:26.535069          198 success


In [13]:
# Cell 10 — Write the data dictionary and view the refresh log

# ── Data dictionary as a markdown file ───────────────────────────────────
data_dict = """# NHS A&E Pipeline — Data Dictionary

**Source:** NHS England A&E Attendances and Emergency Admissions
**URL:** https://www.england.nhs.uk/statistics/statistical-work-areas/ae-waiting-times-and-activity/
**Update frequency:** Monthly (published on the 2nd Thursday of each month, covering the previous month)
**Downloaded by:** Manual download from NHS England website

---

## Table: reporting.ae_monthly

| Field | Type | Description | Derived from | Notes |
|---|---|---|---|---|
| period | DATE | First day of the reporting month e.g. 2026-03-01 | period_raw column in source | Source format is MSitAE-MARCH-2026; parsed in pipeline |
| org_code | VARCHAR | NHS ODS provider code e.g. RRK | Org Code | Used to identify providers consistently across months |
| parent_org | VARCHAR | NHS England regional team | Parent Org | May change if provider moves region |
| org_name | VARCHAR | Provider name as published | Org name | May change following mergers or reconfigurations |
| type1_attendances | INTEGER | Attendances at Type 1 (major) A&E departments | A&E attendances Type 1 | Excludes UTCs and walk-in centres |
| type2_attendances | INTEGER | Attendances at Type 2 (single specialty) A&E departments | A&E attendances Type 2 | |
| other_attendances | INTEGER | Attendances at other departments including UTCs | A&E attendances Other A&E Department | Definition expanded over time — see NHS guidance |
| total_attendances | INTEGER | Sum of Type 1, Type 2 and other attendances | Derived | Calculated in pipeline; not taken from source TOTAL row |
| over_4h_type1 | INTEGER | Type 1 attendances not discharged, admitted or transferred within 4 hours | Attendances over 4hrs Type 1 | |
| over_4h_type2 | INTEGER | Type 2 attendances over 4 hours | Attendances over 4hrs Type 2 | |
| over_4h_other | INTEGER | Other department attendances over 4 hours | Attendances over 4hrs Other Department | |
| total_over_4h | INTEGER | Sum of all attendances over 4 hours | Derived | Calculated in pipeline |
| within_4h_type1 | INTEGER | Type 1 attendances seen within 4 hours | Derived | type1_attendances minus over_4h_type1 |
| four_hour_pct_type1 | DOUBLE | Percentage of Type 1 attendances within 4 hours | Derived | within_4h_type1 / type1_attendances * 100; national target is 95% |
| wait_4_12h_dta | INTEGER | Patients waiting 4-12 hours from decision to admit to admission | Patients who have waited 4-12 hs from DTA to admission | DTA = Decision To Admit |
| wait_12plus_dta | INTEGER | Patients waiting 12 or more hours from decision to admit | Patients who have waited 12+ hrs from DTA to admission | Key measure of hospital flow pressure |
| emergency_admissions_type1 | INTEGER | Emergency admissions arriving via Type 1 A&E | Emergency admissions via A&E - Type 1 | |
| emergency_admissions_type2 | INTEGER | Emergency admissions arriving via Type 2 A&E | Emergency admissions via A&E - Type 2 | |
| emergency_admissions_other | INTEGER | Emergency admissions arriving via other departments | Emergency admissions via A&E - Other A&E department | |
| other_emergency_admissions | INTEGER | Emergency admissions not via A&E | Other emergency admissions | |
| source_file | VARCHAR | Filename this row was loaded from | Pipeline metadata | Use to trace back to original downloaded file |
| loaded_at | TIMESTAMP | When this row was inserted into the pipeline | Pipeline metadata | |

---

## Table: audit.refresh_log

Records every file load attempt.

| Field | Description |
|---|---|
| load_id | Unique identifier for each load run |
| source_file | Filename loaded |
| publication_month | Month covered by the file e.g. MARCH-2026 |
| started_at | When the load began |
| completed_at | When the load finished |
| rows_loaded | Rows inserted into raw.ae_monthly |
| status | success, failed or running |
| notes | Error message if status is failed |

---

## Table: audit.quality_issues

Records every quality problem detected.

| Field | Description |
|---|---|
| issue_id | Unique identifier |
| load_id | Links to refresh_log |
| check_name | Name of the check that fired |
| severity | ERROR or WARNING |
| description | Human-readable description of the problem |
| affected_rows | Number of rows affected |
| detected_at | When the check ran |

---

## Reporting considerations

NHS England notes several factors that affect interpretation of this data:

- **Provider reconfigurations:** Mergers, splits and reclassifications mean that
  changes between months do not always reflect changes in patient activity.
- **Late submissions:** Some providers submit revised figures after initial publication.
  Files marked revised in the filename contain corrected data.
- **February:** Shorter month means lower raw attendance counts are expected
  and should not be interpreted as a drop in demand.
- **Type 3 / Other departments:** The definition of this category has changed
  over time. Long-run comparisons should account for this.
- **TOTAL row:** Each source file contains a pre-calculated TOTAL summary row
  with org_code = TOTAL. This row is excluded from the reporting table.
"""

dict_path = PROJECT_ROOT / "docs" / "data_dictionary.md"
with open(dict_path, "w", encoding="utf-8") as f:
    f.write(data_dict)
print(f"Data dictionary written to: {dict_path}")

# ── View the refresh log ──────────────────────────────────────────────────
print("\nRefresh log:")
refresh_log = con.execute("""
    SELECT
        load_id,
        source_file,
        publication_month,
        started_at,
        rows_loaded,
        status
    FROM audit.refresh_log
    ORDER BY load_id
""").df()
print(refresh_log.to_string(index=False))

Data dictionary written to: C:\Users\q8x7u\Documents\cv\CV\NHS project\nhs-ae-pipeline\docs\data_dictionary.md

Refresh log:
 load_id                                source_file publication_month                 started_at  rows_loaded  status
       1               February-2026-CSV-Dl8t54.csv     FEBRUARY-2026 2026-09-20 13:58:26.430151          198 success
       2 Monthly-AE-December-2025-revised-g9Hy6.csv     DECEMBER-2025 2026-09-20 13:58:26.458256          198 success
       3  Monthly-AE-January-2026-revised-3dker.csv      JANUARY-2026 2026-09-20 13:58:26.489229          198 success
       4   Monthly-AE-March-2026-revised-flkg42.csv        MARCH-2026 2026-09-20 13:58:26.510483          200 success
       5          Monthly-AE-Nov-25-CSV-revised.csv     NOVEMBER-2025 2026-09-20 13:58:26.535069          198 success


In [14]:
con.close()
print("Connection closed")

Connection closed


In [15]:
import subprocess

commands = [
    ["git", "add", "."],
    ["git", "commit", "-m", "Complete pipeline: raw load, quality checks, transform, report and data dictionary"],
]

for cmd in commands:
    result = subprocess.run(cmd, capture_output=True, text=True,
                           cwd=str(PROJECT_ROOT))
    print(result.stdout or result.stderr)


[master 036b5fc] Complete pipeline: raw load, quality checks, transform, report and data dictionary
 7 files changed, 2280 insertions(+), 1 deletion(-)
 create mode 100644 docs/ae_provider_report.csv
 create mode 100644 docs/data_dictionary.md
 create mode 100644 notebooks/01_pipeline.ipynb
 create mode 100644 notebooks/02_report.ipynb
 create mode 100644 notebooks/explore.ipynb
 create mode 100644 src/config.py



In [16]:
readme = """# NHS A&E Data Pipeline

A Python and SQL pipeline for published NHS England A&E statistics, loading monthly
files into DuckDB with repeatable refreshes, data quality checks, a data dictionary
and provider-level reporting.

## What this project demonstrates
- Data ingestion from a real published government dataset
- Three-layer warehouse design: raw → reporting → audit
- Idempotent loads (rerunning the same file does not create duplicates)
- Five automated data quality checks with evidence they catch bad data
- Derived metrics calculated from components rather than trusting source totals
- Data dictionary covering field definitions, source and update frequency
- Refresh log recording every load attempt and outcome

## Architecture
NHS England Website
│
▼ Monthly CSV files
data/raw/
│
▼ load into
raw.ae_monthly (all values stored as VARCHAR — original data preserved)
│
▼ quality checks → audit.quality_issues
reporting.ae_monthly (cleaned, typed, TOTAL rows excluded)
│
▼
docs/ae_provider_report.csv


## How to run

```bash
pip install duckdb pandas openpyxl requests
```

Open `notebooks/01_pipeline.ipynb` and run all cells in order.
Open `notebooks/02_report.ipynb` after closing the pipeline connection.

## Dataset

Five months of data: November 2025 to March 2026.
Source: NHS England A&E Attendances and Emergency Admissions
https://www.england.nhs.uk/statistics/statistical-work-areas/ae-waiting-times-and-activity/

## Key findings (November 2025 – March 2026)

- National Type 1 four-hour performance ranged from 57.0% to 63.9%,
  well below the 95% national standard
- January 2026 had the worst performance (57.0%) and the highest
  12+ hour waits from decision to admit (71,517) — consistent with winter pressure
- March 2026 showed improvement (63.9%) with the fewest 12+ hour waits (46,665)
- University Hospitals Plymouth had the lowest Type 1 performance
  in March 2026 at 37.4%

## Reporting considerations

NHS England notes several factors that affect interpretation:

- **Provider reconfigurations:** Mergers and reclassifications mean changes
  between months do not always reflect changes in patient activity
- **Late submissions:** Files marked revised contain corrected data submitted
  after initial publication — always use the most recent revised file
- **February:** Shorter month means lower raw attendance counts are expected
  and should not be interpreted as a reduction in demand
- **TOTAL row:** Each source file contains a pre-calculated summary row
  (org_code = TOTAL) which is excluded from the reporting table;
  all totals in this pipeline are derived from provider-level rows

## Data quality checks

Five checks run automatically after each load:

| Check | Severity | Description |
|---|---|---|
| MISSING_ORG_CODE | ERROR | Flags rows where org_code is null or blank |
| DUPLICATE_ORG_MONTH | ERROR | Flags duplicate period and org_code combinations |
| NEGATIVE_COUNTS | ERROR | Flags negative attendance values |
| PERFORMANCE_OUT_OF_RANGE | WARNING | Flags four_hour_pct_type1 outside 0–100 |
| WITHIN_4H_EXCEEDS_TOTAL | ERROR | Flags where within_4h_type1 exceeds type1_attendances |

All checks were verified against a deliberately introduced bad row.

## Files

| File | Description |
|---|---|
| notebooks/01_pipeline.ipynb | Full pipeline: load, checks, transform |
| notebooks/02_report.ipynb | National summary and provider report |
| docs/data_dictionary.md | Field definitions, sources and reporting considerations |
| docs/ae_provider_report.csv | Provider-level report output |
| docs/download_log.csv | Record of source files, URLs and download dates |
"""

readme_path = PROJECT_ROOT / "README.md"
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)
print("README.md written")

README.md written
